# Análisis de embudo y retención para MercadoLibre

Este notebook reproduce el análisis principal del proyecto a partir de los CSV procesados incluidos en el repositorio.

## 1. Objetivo

Analizar el embudo de conversión y la retención de usuarios para detectar puntos de fuga, cohortes atípicas y oportunidades de mejora.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path('..') if Path('..').exists() else Path('.')
DATA_DIR = BASE_DIR / 'data' / 'processed'
IMG_DIR = BASE_DIR / 'images'
IMG_DIR.mkdir(exist_ok=True)


In [ ]:
funnel = pd.read_csv(DATA_DIR / 'funnel_general.csv')
funnel_country = pd.read_csv(DATA_DIR / 'funnel_by_country.csv')
ret_country = pd.read_csv(DATA_DIR / 'retention_by_country.csv')
ret_cohort = pd.read_csv(DATA_DIR / 'retention_by_cohort.csv', parse_dates=['cohort'])

display(funnel)


## 2. Análisis del embudo general

In [ ]:
funnel['previous_conversion_pct'] = funnel['conversion_pct'].shift(1)
funnel['relative_drop_pct'] = ((funnel['previous_conversion_pct'] - funnel['conversion_pct']) / funnel['previous_conversion_pct']) * 100
funnel


In [ ]:
max_drop = funnel.dropna(subset=['relative_drop_pct']).sort_values('relative_drop_pct', ascending=False).iloc[0]
print(f"Mayor caída: {max_drop['stage']} con pérdida relativa de {max_drop['relative_drop_pct']:.2f}%")
print(f"Conversión final a compra: {funnel.loc[funnel['stage'] == 'conversion_purchase', 'conversion_pct'].iloc[0]:.2f}%")


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(funnel['stage'].str.replace('conversion_', '').str.replace('_', ' '), funnel['conversion_pct'])
plt.title('Conversión por etapa del embudo')
plt.ylabel('Conversión (%)')
plt.xticks(rotation=35, ha='right')
for i, v in enumerate(funnel['conversion_pct']):
    plt.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


## 3. Conversión por país

In [ ]:
country_purchase = funnel_country[['country', 'conversion_purchase']].sort_values('conversion_purchase', ascending=False)
country_purchase


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(country_purchase['country'], country_purchase['conversion_purchase'])
plt.title('Conversión final a compra por país')
plt.ylabel('Compra final (%)')
plt.xticks(rotation=35, ha='right')
for i, v in enumerate(country_purchase['conversion_purchase']):
    plt.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


## 4. Retención por país

In [ ]:
retention_cols = ['retention_d7_pct', 'retention_d14_pct', 'retention_d21_pct', 'retention_d28_pct']
ret_country[retention_cols].mean().round(2)


In [ ]:
plt.figure(figsize=(10, 5))
for col in retention_cols:
    plt.plot(ret_country['country'], ret_country[col], marker='o', label=col.replace('retention_', '').replace('_pct', '').upper())
plt.title('Retención por país')
plt.ylabel('Retención (%)')
plt.xticks(rotation=35, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Retención por cohorte

In [ ]:
ret_cohort


In [ ]:
plt.figure(figsize=(10, 5))
for col in retention_cols:
    plt.plot(ret_cohort['cohort'].dt.strftime('%Y-%m'), ret_cohort[col], marker='o', label=col.replace('retention_', '').replace('_pct', '').upper())
plt.title('Retención por cohorte')
plt.ylabel('Retención (%)')
plt.xticks(rotation=35, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
heatmap_data = ret_cohort[retention_cols].to_numpy()
plt.figure(figsize=(8, 5))
im = plt.imshow(heatmap_data, aspect='auto')
plt.title('Heatmap de retención por cohorte')
plt.xticks(range(len(retention_cols)), ['D7', 'D14', 'D21', 'D28'])
plt.yticks(range(len(ret_cohort)), ret_cohort['cohort'].dt.strftime('%Y-%m'))
plt.colorbar(im, label='Retención (%)')
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        plt.text(j, i, f'{heatmap_data[i, j]:.1f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 6. Conclusiones

- El principal cuello de botella está entre `select_item` y `add_to_cart`.
- La conversión final a compra es baja, por lo que cualquier mejora en etapas tempranas puede tener impacto acumulado.
- Hay países con conversión final nula que requieren revisión específica.
- La cohorte de agosto 2025 presenta una caída atípica en retención y debe investigarse.